<a href="https://colab.research.google.com/github/heerify15/T5_Text_Summarizer/blob/main/t5_text_summarizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Text Summarizer

## Importing Packages

In [1]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration
import re # Regex
import torch

## Fetching Dataset

In [2]:
train_data = pd.read_csv("samsum-train.csv")
valid_data = pd.read_csv("samsum-validation.csv")

## Collecting Basic Info

In [3]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [4]:
train_data.info()
train_data.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14732 entries, 0 to 14731
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        14732 non-null  object
 1   dialogue  14731 non-null  object
 2   summary   14732 non-null  object
dtypes: object(3)
memory usage: 345.4+ KB


(14732, 3)

In [5]:
valid_data.head()

,id,dialogue,summary
0,13817023,"A: Hi Tom, are you busy tomorrow’s afternoon?\...",A will go to the animal shelter tomorrow to ge...
1,13716628,Emma: I’ve just fallen in love with this adven...,Emma and Rob love the advent calendar. Lauren ...
2,13829420,Jackie: Madison is pregnant\r\nJackie: but she...,Madison is pregnant but she doesn't want to ta...
3,13819648,Marla: <file_photo>\r\nMarla: look what I foun...,Marla found a pair of boxers under her bed.
4,13728448,Robert: Hey give me the address of this music ...,Robert wants Fred to send him the address of t...


In [6]:
valid_data.info()
valid_data.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 818 entries, 0 to 817
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        818 non-null    object
 1   dialogue  818 non-null    object
 2   summary   818 non-null    object
dtypes: object(3)
memory usage: 19.3+ KB


(818, 3)

## Random Sampling

In [7]:
train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
valid_data = valid_data.sample(n=500, random_state=42).reset_index(drop=True)

In [8]:
train_data.info()
train_data.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        4000 non-null   object
 1   dialogue  4000 non-null   object
 2   summary   4000 non-null   object
dtypes: object(3)
memory usage: 93.9+ KB


(4000, 3)

In [9]:
valid_data.info()
valid_data.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        500 non-null    object
 1   dialogue  500 non-null    object
 2   summary   500 non-null    object
dtypes: object(3)
memory usage: 11.8+ KB


(500, 3)

## Data Cleaning

### 1. Check for Missing Values

In [10]:
train_missing = pd.DataFrame({
    "missing_count": train_data.isnull().sum(),
    "missing_%": (train_data.isnull().sum()/len(train_data))*100
})
train_missing = train_missing[train_missing["missing_count"]>0]
print("No missing values found in training dataset!") if train_missing.empty else train_missing

No missing values found in training dataset!


In [11]:
valid_missing = pd.DataFrame({
    "missing_count": valid_data.isnull().sum(),
    "missing_%": (valid_data.isnull().sum()/len(valid_data))*100
})
valid_missing = valid_missing[valid_missing["missing_count"]>0]
print("No missing values found in validation dataset!") if valid_missing.empty else valid_missing

No missing values found in validation dataset!


### 2. Check for Duplicate Values

In [12]:
duplicates = train_data.duplicated().sum()

if duplicates > 0:
    duplicate_rows = train_data[train_data.duplicated()].drop_duplicates()
    print("Rows removed:", duplicates)
    print("Old Shape:", train_data.shape)
    train_data.drop_duplicates(inplace=True)
    train_data.reset_index(drop=True, inplace=True)
    print("New Shape:", train_data.shape)
else:
    print("No duplicates found in training dataset!")

No duplicates found in training dataset!


In [13]:
duplicates = valid_data.duplicated().sum()

if duplicates > 0:
    duplicate_rows = valid_data[valid_data.duplicated()].drop_duplicates()
    print("Rows removed:", duplicates)
    print("Old Shape:", valid_data.shape)
    valid_data.drop_duplicates(inplace=True)
    valid_data.reset_index(drop=True, inplace=True)
    print("New Shape:", valid_data.shape)
else:
    print("No duplicates found in validation dataset!")

No duplicates found in validation dataset!


### 3. Fixing Column names & data

In [14]:
train_data.columns = train_data.columns.str.strip()

for col in train_data.select_dtypes(include="object"):
    train_data[col] = train_data[col].str.strip()

In [15]:
valid_data.columns = valid_data.columns.str.strip()

for col in valid_data.select_dtypes(include="object"):
    valid_data[col] = valid_data[col].str.strip()

## Data Pre-processing

In [16]:
def clean_data(text):
    text = re.sub(r"\r\n", " ", text) # Removing lines
    text = re.sub(r"\s+", " ", text) # Removing extra spaces
    text = re.sub(r"<.*?>", " ", text) # Removing html tags
    text.strip().lower() # Removing trailing spaces and converting to lowercase
    return text

train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

valid_data["dialogue"] = valid_data["dialogue"].apply(clean_data)
valid_data["summary"] = valid_data["summary"].apply(clean_data)

## Tokenization

### 1. Download Tokenizer

In [17]:
tokenizer = T5Tokenizer.from_pretrained("t5-small") # Light weight T5 Model

### 2. Convert raw data => tokens

In [18]:
def tokenize(data):
    inputs = tokenizer(data["dialogue"], padding="max_length", max_length=512, truncation=True)
    targets = tokenizer(data["summary"], padding="max_length", max_length=150, truncation=True)

    inputs["labels"] = targets["input_ids"] # Token ids are added to inputs as labels
    return inputs

train_dataset = train_data.apply(tokenize, axis=1).tolist()
valid_dataset = valid_data.apply(tokenize, axis=1).tolist()

In [19]:
train_dataset[0]

{'input_ids': [28866, 10, 7102, 55, 3, 23, 764, 640, 48, 8513, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 28866, 10, 19542, 10, 2018, 55, 3, 10, 61, 1333, 6, 68, 27, 31, 162, 641, 608, 34, 5, 3, 10, 61, 19542, 10, 299, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [20]:
valid_dataset[0]

{'input_ids': [4857, 26, 10, 2275, 210, 6, 410, 25, 1616, 24, 79, 31, 60, 3, 23560, 178, 12, 3, 9, 315, 3066, 58, 5088, 10, 14228, 9, 9, 9, 9, 144, 3, 10, 32, 5088, 10, 150, 55, 213, 31, 26, 25, 1616, 24, 58, 4857, 26, 10, 168, 6, 34, 31, 7, 882, 2314, 4857, 26, 10, 11825, 131, 1219, 178, 5088, 10, 11, 103, 25, 214, 125, 34, 1112, 21, 178, 58, 4857, 26, 10, 79, 751, 31, 17, 483, 8, 5812, 7, 4857, 26, 10, 68, 3, 23, 214, 8, 16879, 56, 129, 7873, 972, 5088, 10, 11, 3, 23, 3382, 24, 19, 3, 9, 888, 24, 19, 5741, 12, 143, 762, 1842, 4857, 26, 10, 17945, 6, 3382, 78, 4857, 26, 10, 79, 43, 3, 9, 6613, 194, 13, 1705, 3, 31, 235, 143, 378, 1842, 31, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [21]:
# input_ids => tokenized inputs (Here, 1 is for EOS, i.e. end of seq. for summary & 0 is for padding)
# attention_mask => (Here, 1 is for valid token id, & 0 is for padding)
# labels => tokenized summary (target)

## Loading the Pre-trained Model

In [22]:
model = T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

## Fine Tuning the Model

In [23]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Device:", device)
model.to(device)

Device: cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

## Defining the Training Arguments

In [24]:
training_args = TrainingArguments(
    output_dir = "./results",

    num_train_epochs = 6,
    weight_decay = 0.01,

    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 8,

    eval_strategy = "epoch",
    save_strategy = "epoch",

    warmup_steps = 500 # No. of Steps in which the value of learning rate will reach from 0 to it's default value
)

## Defining the Trainer

In [25]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = valid_dataset
)

## Training the Model

In [26]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,4.024957,0.390656
2,0.407535,0.366440
3,0.383335,0.358941
4,0.371631,0.355982
5,0.364639,0.354942
6,0.360650,0.354219


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.9854579010009765, metrics={'train_runtime': 1353.11, 'train_samples_per_second': 17.737, 'train_steps_per_second': 2.217, 'total_flos': 3248203235328000.0, 'train_loss': 0.9854579010009765, 'epoch': 6.0})

## Saving & Loading the Model

In [27]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model/tokenizer_config.json',
 './saved_summary_model/tokenizer.json')

In [28]:
model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model") # To use the saved model
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

## Testing the core logic for summarization

In [29]:
def summarize_dialogue(dialogue):
  # Clean the dialogue
  dialogue = clean_data(dialogue)

  # Tokenize
  inputs = tokenizer(
      dialogue,
      padding="max_length",
      max_length=512,
      truncation=True,
      return_tensors="pt" # pytorch tensors
  ).to(device)

  # Generating Summary => We will get Token ids
  model.to(device)
  generated_ids = model.generate(
      input_ids = inputs["input_ids"],
      attention_mask = inputs["attention_mask"],
      max_length = 150,
      num_beams = 4, # Generates 4 diffierent outputs & chooses the best out of 4
      early_stopping = True
  )

  # Decoding Token ids to get the text summary
  summary = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
  return summary

In [30]:
test_dialogue = """
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. However, this growth has also raised questions about job displacement and ethical concerns.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks such as language understanding, image recognition, and even code generation.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is becoming a key area of research.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies. The goal is to balance innovation with accountability.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as “black boxes,” making it difficult to understand how decisions are made.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, and long-term societal impact.

Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be crucial to ensure that AI systems are developed and used in a safe and beneficial way.
"""

summary = summarize_dialogue(test_dialogue)
print(summary)

Experts are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. Experts highlight the importance of responsible AI development, including data privacy, security, and long-term impact.


## Saving Google Colab Work to Google Drive

In [33]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/T5_Text_Summarizer

# Save training results / checkpoints
!cp -r /content/results /content/drive/MyDrive/T5_Text_Summarizer/

# Save final trained model + tokenizer
trainer.save_model("/content/drive/MyDrive/T5_Text_Summarizer/final_model")
tokenizer.save_pretrained("/content/drive/MyDrive/T5_Text_Summarizer/final_model")

# Save saved_model_summary
!cp -r /content/saved_summary_model /content/drive/MyDrive/T5_Text_Summarizer/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]